In [0]:
# do ENV so it runs wherever
env='d'
# hist date '2025-08-31'(if we have the last rejections)
hist_date='2025-08-31'

In [0]:
#JUST FOR SAFETY PURPOSE FOR FIRST RUN-probably not needed
#delete bkp table
del1 = spark.sql(f""" 
drop table if exists `{env}-cdh-delivery`.tiresias.export_consumer_BEFORE_DELETE

""")

del1 = spark.sql(f""" 
drop table if exists `{env}-cdh-delivery`.tiresias.export_business_BEFORE_DELETE

""")
#------------------------------------------------------
#KEEP BACK UP
bkp1 = spark.sql(f""" 
CREATE OR REPLACE TABLE `{env}-cdh-delivery`.tiresias.export_business_BEFORE_DELETE AS 
SELECT * FROM `{env}-cdh-delivery`.tiresias.export_business

""")

bkp2 = spark.sql(f""" 
CREATE OR REPLACE TABLE `{env}-cdh-delivery`.tiresias.export_consumer_BEFORE_DELETE AS 
SELECT * FROM `{env}-cdh-delivery`.tiresias.export_consumer

""")


In [0]:
B_findings = spark.sql(f""" 
-----------------------------------------------------------------
---checks in hist for extra
WITH CurrentAccounts AS (--get current accounts-cdis and theirs closing info
    SELECT DISTINCT
        `ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` , PORTFOLIO
        ,`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`,`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`
        ,`CDI`
    FROM
        `{env}-cdh-delivery`.tiresias.export_business
),

RankedHistory AS (-- check for hist info for those ^
    SELECT
        hist.*,      now_acc.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ` as now_kleisimo, now_acc.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ` as now_diakopi,now_acc.PORTFOLIO AS PORTFOLIONAME,
           p.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ` as now_kleisimo_onPrem, p.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ` as now_diakopi_onPrem,
        case when p.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` is null then 'not on prem' else null end as exists_onPrem,
        ROW_NUMBER() OVER(
            PARTITION BY hist.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` , hist.`CDI`
            ORDER BY hist.`rpt_date` DESC
        ) as rn
    FROM  `{env}-cdh-delivery`.tiresias.export_business_hist AS hist
    JOIN  CurrentAccounts AS now_acc -- get current accounts
    ON    hist.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` = now_acc.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
    AND hist.`CDI` = now_acc.`CDI`
    left join `{env}-cdh-delivery`.tiresias.business_loadfromonprem P 
    ON    hist.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` = p.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
    AND hist.`CDI` = p.`CDI`
    and p.portfolioid=hist.portfolioid
    where hist.rpt_date <= '{hist_date}'
),
LatestStatus AS (--get only the latest snap
    SELECT * FROM RankedHistory WHERE rn = 1
)
SELECT  --fields to check
    PORTFOLIONAME,
    ls.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    ls.`CDI`,
    ls.rpt_date AS Historical_rpt_date,
    CASE
        WHEN (ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ` IS NOT NULL AND TRIM(LOWER(ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`)) NOT IN ('', '7', 'null'))
            THEN 'Είχε Δείκτη Κλεισίματος'
        WHEN (ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ` IS NOT NULL AND TRIM(LOWER(ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`)) NOT IN ('', 'null'))
            THEN 'Είχε Δείκτη Διακοπής'
    END AS FlagType,
    
    CASE
        WHEN (ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ` IS NOT NULL AND TRIM(LOWER(ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`)) NOT IN ('', '7', 'null'))
            THEN ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`
        WHEN (ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ` IS NOT NULL AND TRIM(LOWER(ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`)) NOT IN ('', 'null'))
            THEN ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`
    END AS FlagValue
   -- ,ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`
    ,now_diakopi,now_kleisimo
    ,now_diakopi_onPrem,now_kleisimo_onPrem
    ,exists_onPrem
    ,ls.has_error
    ,ls.errors
    
FROM
    LatestStatus AS ls
WHERE --DON'T show me if it was not closed or 7 because then i still want to show it on my export
    (   --"Δείκτης Κλεισίματος"
        ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ` IS NOT NULL
        AND TRIM(LOWER(ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`)) NOT IN ('', '7', 'null')
    )
    OR
    (-- "Δείκτης Διακοπής"
        ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ` IS NOT NULL
        AND TRIM(LOWER(ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`)) NOT IN ('', 'null')
    )
ORDER BY
    ls.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    ls.rpt_date DESC;


""")
B_findings.createOrReplaceTempView("B_findings")

In [0]:
#TO BE REMOVED
'''test = spark.sql(f""" 
                 
select count(*) from B_findings

""")
display(test)'''

In [0]:
#filter the cases we want to cut
B_tobeCut = spark.sql(f""" 
                 
select DISTINCT PORTFOLIONAME
       ,`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
       ,CDI
from B_findings
where exists_onPrem='not on prem'
--2252
/*UNION

select DISTINCT PORTFOLIONAME
       ,`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
       ,CDI
from B_findings
where exists_onPrem IS NULL
AND ERRORS IS  NULL*/
--616

""")
B_tobeCut.createOrReplaceTempView("B_tobeCut")
display(B_tobeCut)

In [0]:
#DELETE
#filter the cases we want to cut
B_DELETE=spark.sql(f""" MERGE INTO `{env}-cdh-delivery`.tiresias.export_business AS t USING (
					    	SELECT  *
					    	FROM B_tobeCut 
						)s						
                           -- INNER JOIN `{env}-cdh-delivery`.tiresias.export_business s-- to be removed
                           -- ON s.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` = t.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` AND t.CDI  = s.CDI 
                           -- AND T.PORTFOLIONAME=s.PORTFOLIO--to be removed
                        ON (
						s.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` = t.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
						AND s.CDI = t.CDI
						AND s.PORTFOLIONAME = t.PORTFOLIO
						)
					    WHEN MATCHED THEN
    					DELETE;
    """)
B_DELETE.createOrReplaceTempView("B_DELETE")
display(B_DELETE) 

In [0]:
C_findings = spark.sql(f""" 
-----------------------------------------------------------------
---checks in hist for extra
WITH CurrentAccounts AS (--get current accounts-cdis and theirs closing info
    SELECT DISTINCT
        `ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` , PORTFOLIO
        ,`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`,`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ_OVERDRAFT`
        ,`CDI`
    FROM
        `{env}-cdh-delivery`.tiresias.export_consumer
),

RankedHistory AS (-- check for hist info for those ^
    SELECT
        hist.*,      now_acc.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ` as now_kleisimo, now_acc.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ_OVERDRAFT` as now_diakopi,now_acc.PORTFOLIO AS PORTFOLIONAME,
           p.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ` as now_kleisimo_onPrem, p.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ`as now_diakopi_onPrem,
        case when p.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` is null then 'not on prem' else null end as exists_onPrem,
        ROW_NUMBER() OVER(
            PARTITION BY hist.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` , hist.`CDI`
            ORDER BY hist.`rpt_date` DESC
        ) as rn
    FROM  `{env}-cdh-delivery`.tiresias.export_consumer_hist AS hist
    JOIN  CurrentAccounts AS now_acc -- get current accounts
    ON    hist.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` = now_acc.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
    AND hist.`CDI` = now_acc.`CDI`
    left join `{env}-cdh-delivery`.tiresias.consumer_loadfromonprem P 
    ON    hist.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` = p.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
    AND hist.`CDI` = p.`CDI`
    and p.portfolioid=hist.portfolioid
    where hist.rpt_date <= '2025-08-31'
),
LatestStatus AS (--get only the latest snap
    SELECT * FROM RankedHistory WHERE rn = 1
)
SELECT  --fields to check
    PORTFOLIONAME,
    ls.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    ls.`CDI`,
    ls.rpt_date AS Historical_rpt_date,
    CASE
        WHEN (ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ` IS NOT NULL AND TRIM(LOWER(ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`)) NOT IN ('', '7', 'null'))
            THEN 'Είχε Δείκτη Κλεισίματος'
        WHEN (ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ` IS NOT NULL AND TRIM(LOWER(ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ`)) NOT IN ('', 'null'))
            THEN 'Είχε Δείκτη Διακοπής'
    END AS FlagType,
    
    CASE
        WHEN (ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ` IS NOT NULL AND TRIM(LOWER(ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`)) NOT IN ('', '7', 'null'))
            THEN ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`
        WHEN (ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ` IS NOT NULL AND TRIM(LOWER(ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ`)) NOT IN ('', 'null'))
            THEN ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ`
    END AS FlagValue
   -- ,ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`
    ,now_diakopi,now_kleisimo
    ,now_diakopi_onPrem,now_kleisimo_onPrem
    ,exists_onPrem
    ,ls.has_error
    ,ls.errors
    
FROM
    LatestStatus AS ls
WHERE --DON'T show me if it was not closed or 7 because then i still want to show it on my export
    (   --"Δείκτης Κλεισίματος"
        ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ` IS NOT NULL
        AND TRIM(LOWER(ls.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`)) NOT IN ('', '7', 'null')
    )
    OR
    (-- "Δείκτης Διακοπής"
        ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ` IS NOT NULL
        AND TRIM(LOWER(ls.`ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ`)) NOT IN ('', 'null')
    )
ORDER BY
    ls.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    ls.rpt_date DESC;


""")
C_findings.createOrReplaceTempView("C_findings")

In [0]:
#TO BE REMOVED
'''test = spark.sql(f""" 
                 
select count(*)
from C_findings--

""")
display(test)'''

In [0]:
#filter the cases we want to cut
C_tobeCut = spark.sql(f""" 
                 
select DISTINCT PORTFOLIONAME
       ,`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
       ,CDI
from C_findings
where exists_onPrem='not on prem'

""")
C_tobeCut.createOrReplaceTempView("C_tobeCut")
display(C_tobeCut)

In [0]:
#DELETE
#filter the cases we want to cut
C_DELETE=spark.sql(f""" MERGE INTO `{env}-cdh-delivery`.tiresias.export_consumer AS t USING (
					    	SELECT  *
					    	FROM C_tobeCut t
							)s
                            --INNER JOIN `{env}-cdh-delivery`.tiresias.export_consumer s-- to be removed
                            --ON s.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` = t.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` AND t.CDI  = s.CDI 
                            --AND T.PORTFOLIONAME=s.PORTFOLIO--to be removed
                            ON (
							s.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` = t.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
							AND s.CDI = t.CDI
							AND s.PORTFOLIONAME = t.PORTFOLIO
							)
					    WHEN MATCHED THEN
    					DELETE;
    """)
B_DELETE.createOrReplaceTempView("C_DELETE")
display(C_DELETE) 